In [8]:
import os
import numpy as np
import pymrio

# -----------------------------
# 0) CONFIG
# -----------------------------
path_zip = r"C:\Users\Dyde-nairn\Documents\python\_input_output_dependencies\IOT_2022_pxp.v3.8.2.zip"

sector_target = "Aluminium ores and concentrates"   # CHANGE if needed
shock_size = 0.1   # 10% shock


# -----------------------------
# 1) LOAD EXIOBASE
# -----------------------------
if not os.path.exists(path_zip):
    raise FileNotFoundError(f"EXIOBASE file not found at: {path_zip}")

print("Loading EXIOBASE...")
exio3 = pymrio.parse_exiobase3(path=path_zip)


print("Calculating system...")
exio3.calc_all()   # ✅ ADD THIS


print("Extracting matrices...")

# Core matrices
Z = exio3.Z.values
X = exio3.x.values.flatten()

# -----------------------------
# BUILD B MATRIX (manually)
# -----------------------------
print("Building B matrix...")

X_safe = X.copy()
X_safe[X_safe == 0] = 1e-12

B = Z / X_safe[:, None]

# -----------------------------
# BUILD GHOSH INVERSE
# -----------------------------
print("Building Ghosh inverse...")

I = np.eye(len(X))

G = np.linalg.inv(I - B)

# -----------------------------
# VALUE ADDED (consistent)
# -----------------------------
v = X - Z.sum(axis=1)



Loading EXIOBASE...
Calculating system...
Extracting matrices...
Building B matrix...
Building Ghosh inverse...


In [6]:
# -----------------------------
# 2) FIND INDEX FUNCTION
# -----------------------------
def find_index(country, sector):
    for i in range(len(country_of)):
        if country_of[i] == country and sector_of[i] == sector:
            return i
    raise ValueError(f"Not found: {country}, {sector}")


# -----------------------------
# 3) SELECT SHOCK NODES
# -----------------------------
print("Selecting China and US nodes...")

idx_china = find_index("CN", sector_target)
idx_us    = find_index("US", sector_target)


# -----------------------------
# 4) APPLY GHOSH SHOCK
# -----------------------------
print("Running Ghosh shock...")

dv = np.zeros_like(v)
dv[idx_china] = -shock_size * v[idx_china]

dX = G @ dv


# -----------------------------
# 5) REALLOCATION MECHANISM
# -----------------------------
print("Running reallocation mechanism...")

NS = len(X)

# US loss (positive)
us_loss = -dX[idx_us]

realloc = np.zeros(NS)

if us_loss > 0:

    # Alternative suppliers (same sector, not China)
    suppliers = np.array([
        i for i in range(NS)
        if sector_of[i] == sector_target and country_of[i] != "CN"
    ])

    # Shares going to US
    B_to_us = B[suppliers, idx_us]
    total = B_to_us.sum()

    if total > 0:
        alpha_tilde = B_to_us / total

        # Extra demand from US
        extra_demand = alpha_tilde * us_loss

        # Vectorized reallocation
        extra_vec = np.zeros(NS)
        extra_vec[suppliers] = extra_demand

        realloc = extra_vec @ B


# -----------------------------
# 6) TOTAL EFFECT
# -----------------------------
total_effect = dX - realloc


# -----------------------------
# 7) AGGREGATE TO COUNTRY
# -----------------------------
def aggregate_by_country(values):
    result = {}
    for c in np.unique(country_of):
        result[c] = values[country_of == c].sum()
    return result


direct_country  = aggregate_by_country(dX)
realloc_country = aggregate_by_country(realloc)
total_country   = aggregate_by_country(total_effect)


# -----------------------------
# 8) PRINT RESULTS
# -----------------------------
print("\n=== RESULTS ===")

def print_country(c):
    print(f"\nCountry: {c}")
    print(f"Direct effect   : {direct_country.get(c, 0):.4f}")
    print(f"Realloc effect  : {-realloc_country.get(c, 0):.4f}")
    print(f"Total effect    : {total_country.get(c, 0):.4f}")

print_country("FR")
print_country("US")
print_country("CN")


# -----------------------------
# 9) SANITY CHECKS
# -----------------------------
print("\n=== SANITY CHECKS ===")

print("US loss (direct):", us_loss)
print("Total realloc (should ~ US loss):", realloc.sum())

# ✅ Check IO identity consistency
v_check = X - Z.sum(axis=1)
print("v consistency check (should be True):", np.allclose(v, v_check))

# ✅ Check positivity
print("All v positive:", np.all(v >= -1e-8))

Loading EXIOBASE...
Extracting matrices...


AttributeError: 'NoneType' object has no attribute 'values'

In [13]:
# -----------------------------
# 2) FIND INDEX FUNCTION (FAST + SAFE)
# -----------------------------
def find_index(country, sector):
    mask = (country_of == country) & (sector_of == sector)
    idx = np.where(mask)[0]
    
    if len(idx) == 0:
        raise ValueError(f"Not found: {country}, {sector}")
    
    return idx[0]


# -----------------------------
# 3) SELECT SHOCK + SOLAR NODES
# -----------------------------
print("Selecting nodes...")

# Shock nodes
idx_china = find_index("CN", sector_target)
idx_us = find_index("US", sector_target)

# Solar nodes
solar_sector = "Electricity by solar photovoltaic"

idx_fr_solar = find_index("FR", solar_sector)
idx_us_solar = find_index("US", solar_sector)


# -----------------------------
# 4) APPLY GHOSH SHOCK
# -----------------------------
print("Running Ghosh shock...")

dv = np.zeros_like(v)
dv[idx_china] = -shock_size * v[idx_china]

dX = G @ dv


# -----------------------------
# 5) REALLOCATION MECHANISM
# -----------------------------
print("Running reallocation mechanism...")

NS = len(X)

us_loss = -dX[idx_us]   # Positive number

realloc = np.zeros(NS)

if us_loss > 0:

    suppliers = np.where(
        (sector_of == sector_target) & (country_of != "CN")
    )[0]

    if len(suppliers) > 0:
        B_to_us = B[suppliers, idx_us]
        total = B_to_us.sum()

        if total > 1e-12:
            alpha_tilde = B_to_us / total

            extra_demand = alpha_tilde * us_loss

            extra_vec = np.zeros(NS)
            extra_vec[suppliers] = extra_demand

            realloc = extra_vec @ B


# -----------------------------
# 6) TOTAL EFFECT
# -----------------------------
total_effect = dX - realloc


# -----------------------------
# 7) SOLAR-SPECIFIC RESULTS (CORE OUTPUT)
# -----------------------------
print("\n=== SOLAR SECTOR RESULTS ===")

def print_solar(country, idx):
    print(f"\nCountry: {country} (Solar)")
    print(f"Direct effect   : {dX[idx]:.10f}")
    print(f"Realloc effect  : {-realloc[idx]:.10f}")
    print(f"Total effect    : {total_effect[idx]:.10f}")

print_solar("FR", idx_fr_solar)
print_solar("US", idx_us_solar)


# -----------------------------
# 8) NORMALIZED (% CHANGE)  ← MOST IMPORTANT
# -----------------------------
print("\n=== SOLAR IMPACT (% CHANGE) ===")

def print_solar_pct(country, idx):
    pct_total = 100 * total_effect[idx] / X[idx]
    pct_direct = 100 * dX[idx] / X[idx]
    pct_realloc = 100 * (-realloc[idx]) / X[idx]

    print(f"\n{country}:")
    print(f"  Direct  : {pct_direct:.6f}%")
    print(f"  Realloc : {pct_realloc:.6f}%")
    print(f"  Total   : {pct_total:.6f}%")

print_solar_pct("FR", idx_fr_solar)
print_solar_pct("US", idx_us_solar)


# -----------------------------
# 9) TOTAL ECONOMY (REFERENCE ONLY)
# -----------------------------
def aggregate_by_country(values):
    result = {}
    for c in np.unique(country_of):
        result[c] = values[country_of == c].sum()
    return result

direct_country = aggregate_by_country(dX)
realloc_country = aggregate_by_country(realloc)
total_country  = aggregate_by_country(total_effect)

print("\n=== TOTAL ECONOMY EFFECT (REFERENCE) ===")

for c in ["FR", "US", "CN"]:
    print(f"\nCountry: {c}")
    print(f"Direct   : {direct_country.get(c, 0):.6f}")
    print(f"Realloc  : {-realloc_country.get(c, 0):.6f}")
    print(f"Total    : {total_country.get(c, 0):.6f}")


# -----------------------------
# 10) SANITY CHECKS
# -----------------------------
print("\n=== SANITY CHECKS ===")

print("US loss (direct):", us_loss)
print("Total realloc (~ US loss):", realloc.sum())

# IO identity check
v_check = X - Z.sum(axis=1)
print("v consistency:", np.allclose(v, v_check))

# Numerical stability check
v_min = v.min()
print("Minimum v value:", v_min)
print("All v approx positive:", v_min > -1e-8)

Selecting nodes...
Running Ghosh shock...
Running reallocation mechanism...

=== SOLAR SECTOR RESULTS ===

Country: FR (Solar)
Direct effect   : -0.0000018394
Realloc effect  : -0.0000000000
Total effect    : -0.0000018394

Country: US (Solar)
Direct effect   : -0.0000008159
Realloc effect  : -0.0000000000
Total effect    : -0.0000008159

=== SOLAR IMPACT (% CHANGE) ===

FR:
  Direct  : -0.000005%
  Realloc : -0.000000%
  Total   : -0.000005%

US:
  Direct  : -0.000001%
  Realloc : -0.000000%
  Total   : -0.000001%

=== TOTAL ECONOMY EFFECT (REFERENCE) ===

Country: FR
Direct   : -0.000566
Realloc  : -0.000000
Total    : -0.000566

Country: US
Direct   : -0.000573
Realloc  : -0.000004
Total    : -0.000577

Country: CN
Direct   : -0.489666
Realloc  : -0.000000
Total    : -0.489666

=== SANITY CHECKS ===
US loss (direct): 5.5226276919879556e-06
Total realloc (~ US loss): 5.3918359888289855e-06
v consistency: True
Minimum v value: -7.986818673089147e-08
All v approx positive: False


In [27]:
# ============================================================
# PARAMETERS
# ============================================================

theta = 1        # ✅ substitutability (0 = none, 1 = full)
shock_size = 1.0   # size of China shock
sector_target = "Other non-ferrous metal ores and concentrates"

# ============================================================
# 1) FIND INDEX FUNCTION
# ============================================================

def find_index(country, sector):
    mask = (country_of == country) & (sector_of == sector)
    idx = np.where(mask)[0]
    
    if len(idx) == 0:
        raise ValueError(f"Not found: {country}, {sector}")
    
    return idx[0]


# ============================================================
# 2) SELECT NODES
# ============================================================

print("Selecting nodes...")

# Shock sector
idx_china = find_index("CN", sector_target)
idx_us    = find_index("US", sector_target)

# Solar sector
solar_sector = "Electricity by solar photovoltaic"

idx_fr_solar = find_index("FR", solar_sector)
idx_us_solar = find_index("US", solar_sector)


# ============================================================
# 3) INITIAL GHOSH SHOCK
# ============================================================

print("Running initial Ghosh shock...")

dv = np.zeros_like(v)
dv[idx_china] = -shock_size * v[idx_china]

dX = G @ dv


# ============================================================
# 4) SUPPLIER SUBSTITUTION (PANON-STYLE)
# ============================================================

print("Running supplier substitution...")

NS = len(X)

us_loss = -dX[idx_us]   # positive

realloc_losses = np.zeros(NS)
dX_recovery     = np.zeros(NS)

if us_loss > 0:

    # -----------------------------
    # A) Identify alternative suppliers
    # -----------------------------
    suppliers = np.where(
        (sector_of == sector_target) & (country_of != "CN")
    )[0]

    if len(suppliers) > 0:

        # Existing shares supplying US
        B_to_us = B[suppliers, idx_us]
        total = B_to_us.sum()

        if total > 1e-12:

            # -----------------------------
            # B) Substitution intensity (θ)
            # -----------------------------
            alpha_tilde = B_to_us / total

            # ✅ Panon-style: partial substitution
            extra_demand = theta * alpha_tilde * us_loss

            # -----------------------------
            # C) CROWDING-OUT
            # -----------------------------
            extra_vec = np.zeros(NS)
            extra_vec[suppliers] = extra_demand

            realloc_losses = extra_vec @ B

            # -----------------------------
            # D) US RECOVERY
            # -----------------------------
            us_input_gain = extra_demand.sum()

            dv_recovery = np.zeros_like(v)
            dv_recovery[idx_us] = us_input_gain

            dX_recovery = G @ dv_recovery


# ============================================================
# 5) TOTAL EFFECT
# ============================================================

total_effect = dX - realloc_losses + dX_recovery


# ============================================================
# 6) SOLAR RESULTS (CORE OUTPUT)
# ============================================================

print("\n=== SOLAR SECTOR RESULTS ===")

def print_solar(country, idx):
    print(f"\nCountry: {country} (Solar)")
    print(f"Direct effect   : {dX[idx]:.10f}")
    print(f"Realloc effect  : {-realloc_losses[idx]:.10f}")
    print(f"Recovery effect : {dX_recovery[idx]:.10f}")
    print(f"Total effect    : {total_effect[idx]:.10f}")

print_solar("FR", idx_fr_solar)
print_solar("US", idx_us_solar)


# ============================================================
# 7) NORMALIZED (% CHANGE)
# ============================================================

print("\n=== SOLAR IMPACT (% CHANGE) ===")

def print_solar_pct(country, idx):
    pct_direct  = 100 * dX[idx] / X[idx]
    pct_realloc = 100 * (-realloc_losses[idx]) / X[idx]
    pct_recover = 100 * dX_recovery[idx] / X[idx]
    pct_total   = 100 * total_effect[idx] / X[idx]

    print(f"\n{country}:")
    print(f"  Direct   : {pct_direct:.6f}%")
    print(f"  Realloc  : {pct_realloc:.6f}%")
    print(f"  Recovery : {pct_recover:.6f}%")
    print(f"  Total    : {pct_total:.6f}%")

print_solar_pct("FR", idx_fr_solar)
print_solar_pct("US", idx_us_solar)


# ============================================================
# 8) TOTAL ECONOMY (REFERENCE)
# ============================================================

def aggregate_by_country(values):
    result = {}
    for c in np.unique(country_of):
        result[c] = values[country_of == c].sum()
    return result

direct_country  = aggregate_by_country(dX)
realloc_country = aggregate_by_country(realloc_losses)
recover_country = aggregate_by_country(dX_recovery)
total_country   = aggregate_by_country(total_effect)

print("\n=== TOTAL ECONOMY EFFECT ===")

for c in ["FR", "US", "CN"]:
    print(f"\nCountry: {c}")
    print(f"Direct   : {direct_country.get(c, 0):.6f}")
    print(f"Realloc  : {-realloc_country.get(c, 0):.6f}")
    print(f"Recovery : {recover_country.get(c, 0):.6f}")
    print(f"Total    : {total_country.get(c, 0):.6f}")


# ============================================================
# 9) SANITY CHECKS
# ============================================================

print("\n=== SANITY CHECKS ===")

print("US loss (initial):", us_loss)
print("Total realloc (~ θ × US loss):", realloc_losses.sum())
print("Total recovery (US):", dX_recovery[idx_us])

v_check = X - Z.sum(axis=1)
print("v consistency:", np.allclose(v, v_check))
print("Min v:", v.min())


Selecting nodes...
Running initial Ghosh shock...
Running supplier substitution...

=== SOLAR SECTOR RESULTS ===

Country: FR (Solar)
Direct effect   : -0.0994670527
Realloc effect  : -0.0000000000
Recovery effect : 0.0000002700
Total effect    : -0.0994667827

Country: US (Solar)
Direct effect   : -0.0422681727
Realloc effect  : -0.0000000505
Recovery effect : 0.0000209885
Total effect    : -0.0422472348

=== SOLAR IMPACT (% CHANGE) ===

FR:
  Direct   : -0.285554%
  Realloc  : -0.000000%
  Recovery : 0.000001%
  Total    : -0.285553%

US:
  Direct   : -0.040341%
  Realloc  : -0.000000%
  Recovery : 0.000020%
  Total    : -0.040321%

=== TOTAL ECONOMY EFFECT ===

Country: FR
Direct   : -18.386196
Realloc  : -0.002852
Recovery : 0.000064
Total    : -18.388984

Country: US
Direct   : -38.062572
Realloc  : -1.019506
Recovery : 1.346838
Total    : -37.735240

Country: CN
Direct   : -1435.385845
Realloc  : -0.055556
Recovery : 0.000041
Total    : -1435.441359

=== SANITY CHECKS ===
US loss

In [29]:
# ============================================================
# PARAMETERS
# ============================================================

shock_size = 1.0


# ============================================================
# 1) FIND INDEX FUNCTION
# ============================================================

def find_index(country, sector):
    mask = (country_of == country) & (sector_of == sector)
    idx = np.where(mask)[0]
    
    if len(idx) == 0:
        raise ValueError(f"Not found: {country}, {sector}")
    
    return idx[0]


# ============================================================
# 2) SELECT NODES
# ============================================================

print("Selecting nodes...")

# Shock sector
idx_china = find_index("CN", sector_target)
idx_us    = find_index("US", sector_target)

# Solar nodes
solar_sector = "Electricity by solar photovoltaic"

idx_fr_solar = find_index("FR", solar_sector)
idx_us_solar = find_index("US", solar_sector)


# ============================================================
# 3) INITIAL GHOSH SHOCK
# ============================================================

print("Running initial Ghosh shock...")

dv = np.zeros_like(v)
dv[idx_china] = -shock_size * v[idx_china]

dX = G @ dv


# ============================================================
# 4) TARGETED REALLOCATION (US SOLAR FIXED)
# ============================================================

print("Running targeted reallocation (US solar fixed)...")

NS = len(X)

# -----------------------------
# A) US solar loss
# -----------------------------
solar_loss_us = -dX[idx_us_solar]

realloc_losses = np.zeros(NS)
dX_recovery     = np.zeros(NS)

if solar_loss_us > 0:

    # -----------------------------
    # B) Identify ALL suppliers to US solar
    # -----------------------------
    suppliers = np.where(B[:, idx_us_solar] > 1e-12)[0]

    if len(suppliers) > 0:

        weights = B[suppliers, idx_us_solar]
        total = weights.sum()

        if total > 1e-12:

            alpha = weights / total

            # -----------------------------
            # C) US pulls inputs to keep solar constant
            # -----------------------------
            extra_demand = alpha * solar_loss_us

            extra_vec = np.zeros(NS)
            extra_vec[suppliers] = extra_demand

            # -----------------------------
            # D) Crowding-out effect
            # -----------------------------
            realloc_losses = extra_vec @ B

            # -----------------------------
            # E) Enforce US solar recovery
            # -----------------------------
            dX_recovery[idx_us_solar] = solar_loss_us


# ============================================================
# 5) TOTAL EFFECT
# ============================================================

total_effect = dX - realloc_losses + dX_recovery


# ============================================================
# 6) SOLAR RESULTS
# ============================================================

print("\n=== SOLAR SECTOR RESULTS ===")

def print_solar(country, idx):
    print(f"\nCountry: {country} (Solar)")
    print(f"Direct effect   : {dX[idx]:.10f}")
    print(f"Realloc effect  : {-realloc_losses[idx]:.10f}")
    print(f"Recovery effect : {dX_recovery[idx]:.10f}")
    print(f"Total effect    : {total_effect[idx]:.10f}")

print_solar("FR", idx_fr_solar)
print_solar("US", idx_us_solar)


# ============================================================
# 7) NORMALIZED (% CHANGE)
# ============================================================

print("\n=== SOLAR IMPACT (% CHANGE) ===")

def print_solar_pct(country, idx):
    pct_direct  = 100 * dX[idx] / X[idx]
    pct_realloc = 100 * (-realloc_losses[idx]) / X[idx]
    pct_recover = 100 * dX_recovery[idx] / X[idx]
    pct_total   = 100 * total_effect[idx] / X[idx]

    print(f"\n{country}:")
    print(f"  Direct   : {pct_direct:.6f}%")
    print(f"  Realloc  : {pct_realloc:.6f}%")
    print(f"  Recovery : {pct_recover:.6f}%")
    print(f"  Total    : {pct_total:.6f}%")

print_solar_pct("FR", idx_fr_solar)
print_solar_pct("US", idx_us_solar)

# ============================================================
# 8) GLOBAL SOLAR IMPACT (FILTERED)
# ============================================================

print("\n=== GLOBAL SOLAR IMPACT (ALL COUNTRIES, CLEANED) ===")

results = []

for c in np.unique(country_of):
    try:
        idx = find_index(c, solar_sector)
    except:
        continue
    
    # ✅ skip tiny sectors
    if X[idx] < 1e-6:
        continue
    
    pct_total   = 100 * total_effect[idx] / X[idx]
    pct_direct  = 100 * dX[idx] / X[idx]
    pct_realloc = 100 * (-realloc_losses[idx]) / X[idx]
    
    results.append((c, pct_total, pct_direct, pct_realloc))


# sort
results_sorted = sorted(results, key=lambda x: x[1])

for c, tot, direct, realloc in results_sorted:
    print(f"{c:3s} | Total: {tot: .6f}% | Direct: {direct: .6f}% | Realloc: {realloc: .6f}%")



# ============================================================
# 8) TOTAL ECONOMY EFFECT
# ============================================================

def aggregate_by_country(values):
    result = {}
    for c in np.unique(country_of):
        result[c] = values[country_of == c].sum()
    return result

direct_country  = aggregate_by_country(dX)
realloc_country = aggregate_by_country(realloc_losses)
recover_country = aggregate_by_country(dX_recovery)
total_country   = aggregate_by_country(total_effect)

print("\n=== TOTAL ECONOMY EFFECT ===")

for c in ["FR", "US", "CN"]:
    print(f"\nCountry: {c}")
    print(f"Direct   : {direct_country.get(c, 0):.6f}")
    print(f"Realloc  : {-realloc_country.get(c, 0):.6f}")
    print(f"Recovery : {recover_country.get(c, 0):.6f}")
    print(f"Total    : {total_country.get(c, 0):.6f}")


# ============================================================
# 9) SANITY CHECKS
# ============================================================

print("\n=== SANITY CHECKS ===")

print("US solar loss (initial):", solar_loss_us)
print("Total realloc pulled:", realloc_losses.sum())
print("US solar recovery:", dX_recovery[idx_us_solar])

# IO identity
v_check = X - Z.sum(axis=1)
print("v consistency:", np.allclose(v, v_check))

print("Minimum v:", v.min())

Selecting nodes...
Running initial Ghosh shock...
Running targeted reallocation (US solar fixed)...

=== SOLAR SECTOR RESULTS ===

Country: FR (Solar)
Direct effect   : -0.0994670527
Realloc effect  : -0.0000000769
Recovery effect : 0.0000000000
Total effect    : -0.0994671296

Country: US (Solar)
Direct effect   : -0.0422681727
Realloc effect  : -0.0000001216
Recovery effect : 0.0422681727
Total effect    : -0.0000001216

=== SOLAR IMPACT (% CHANGE) ===

FR:
  Direct   : -0.285554%
  Realloc  : -0.000000%
  Recovery : 0.000000%
  Total    : -0.285554%

US:
  Direct   : -0.040341%
  Realloc  : -0.000000%
  Recovery : 0.040341%
  Total    : -0.000000%

=== GLOBAL SOLAR IMPACT (ALL COUNTRIES, CLEANED) ===
BR  | Total: -2423109.695282% | Direct: -2423109.695282% | Realloc: -0.000000%
PL  | Total: -599002.712832% | Direct: -599002.712832% | Realloc: -0.000000%
EE  | Total: -315339.839952% | Direct: -315339.839952% | Realloc: -0.000000%
LT  | Total: -3250.511023% | Direct: -3250.511023% | R

In [30]:
# ============================================================
# PARAMETERS
# ============================================================

shock_all = True   # full cutoff China -> US


# ============================================================
# 1) FIND INDEX FUNCTION
# ============================================================

def find_index(country, sector):
    mask = (country_of == country) & (sector_of == sector)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        raise ValueError(f"Not found: {country}, {sector}")
    return idx[0]


# ============================================================
# 2) INITIAL STATE
# ============================================================

print("Preparing baseline...")

Z_new = Z.copy()   # working IO matrix
NS = len(X)

# track crowding-out
realloc_losses = np.zeros(NS)


# ============================================================
# 3) CUT CHINA → US FLOWS
# ============================================================

print("Applying China -> US supply cut...")

lost_inputs_US = np.zeros(NS)

for i in range(NS):
    if country_of[i] == "CN":
        for j in range(NS):
            if country_of[j] == "US":
                
                # record lost flow
                val = Z[i, j]
                lost_inputs_US[j] += val
                
                # cut the flow
                Z_new[i, j] = 0.0


# ============================================================
# 4) REALLOCATION (CORE MECHANISM)
# ============================================================

print("Reallocating US demand across remaining suppliers...")

for j in range(NS):
    
    if country_of[j] != "US":
        continue
    
    lost = lost_inputs_US[j]
    
    if lost <= 1e-12:
        continue
    
    # identify non-China suppliers
    suppliers = [i for i in range(NS)
                 if (country_of[i] != "CN") and (Z[i, j] > 0)]
    
    if len(suppliers) == 0:
        continue
    
    weights = np.array([Z[i, j] for i in suppliers])
    weights = weights / weights.sum()
    
    extra_demand = weights * lost
    
    # apply reallocation
    for k, i in enumerate(suppliers):
        
        # how supplier i distributes output globally
        total_output = Z[:, i].sum()
        
        if total_output <= 1e-12:
            continue
        
        shares = Z[:, i] / total_output
        
        reduction = extra_demand[k] * shares
        
        # reduce supply to all destinations
        Z_new[:, i] -= reduction
        
        # give extra to US
        Z_new[i, j] += extra_demand[k]
        
        # track crowding-out
        realloc_losses += reduction


# ============================================================
# 5) RECOMPUTE OUTPUT
# ============================================================

print("Recomputing production...")

X_new = Z_new.sum(axis=1) + v

delta_X = X_new - X


# ============================================================
# 6) SOLAR RESULTS (KEY OUTPUT)
# ============================================================

print("\n=== SOLAR IMPACT (ALL COUNTRIES, CLEANED) ===")

solar_sector = "Electricity by solar photovoltaic"

results = []

for c in np.unique(country_of):
    try:
        idx = find_index(c, solar_sector)
    except:
        continue
    
    # avoid exploding % from tiny base
    if X[idx] < 1e-6:
        continue
    
    pct_total   = 100 * delta_X[idx] / X[idx]
    
    # approximate direct = before realloc (optional)
    pct_realloc = 100 * (-realloc_losses[idx]) / X[idx]
    
    results.append((c, pct_total, pct_realloc))


# sort countries by solar impact
results_sorted = sorted(results, key=lambda x: x[1])

for c, tot, realloc in results_sorted:
    print(f"{c:3s} | Solar change: {tot: .6f}% | Realloc: {realloc: .6f}%")


# ============================================================
# 7) WHERE CROWDING-OUT HAPPENS (IMPORTANT)
# ============================================================

print("\n=== TOP REALLOCATION LOSSES (ALL SECTORS) ===")

pairs = []

for i in range(NS):
    loss = -realloc_losses[i]
    if loss > 1e-6:
        pairs.append((loss, country_of[i], sector_of[i]))

pairs_sorted = sorted(pairs, reverse=True)

for val, c, s in pairs_sorted[:20]:
    print(f"{c} | {s} | realloc loss = {val:.6f}")


# ============================================================
# 8) COUNTRY-LEVEL TOTAL EFFECT
# ============================================================

print("\n=== TOTAL ECONOMY EFFECT ===")

for c in np.unique(country_of):
    
    mask = (country_of == c)
    
    if X[mask].sum() < 1e-6:
        continue
    
    total_pct = 100 * delta_X[mask].sum() / X[mask].sum()
    realloc_pct = 100 * (-realloc_losses[mask].sum()) / X[mask].sum()
    
    print(f"{c} | Total: {total_pct: .4f}% | Realloc: {realloc_pct: .4f}%")

Preparing baseline...
Applying China -> US supply cut...
Reallocating US demand across remaining suppliers...
Recomputing production...

=== SOLAR IMPACT (ALL COUNTRIES, CLEANED) ===
US  | Solar change: -0.456296% | Realloc: -1.156622%
CA  | Solar change: -0.172020% | Realloc: -0.172020%
MX  | Solar change: -0.161374% | Realloc: -0.161374%
WA  | Solar change: -0.088579% | Realloc: -0.088579%
IE  | Solar change: -0.058423% | Realloc: -0.058423%
CH  | Solar change: -0.042349% | Realloc: -0.042349%
TW  | Solar change: -0.041577% | Realloc: -0.041577%
GB  | Solar change: -0.039582% | Realloc: -0.039582%
BR  | Solar change: -0.034055% | Realloc: -0.034055%
PT  | Solar change: -0.030810% | Realloc: -0.030810%
SI  | Solar change: -0.030355% | Realloc: -0.030355%
DE  | Solar change: -0.030037% | Realloc: -0.030037%
FR  | Solar change: -0.027566% | Realloc: -0.027566%
GR  | Solar change: -0.022138% | Realloc: -0.022138%
WL  | Solar change: -0.021021% | Realloc: -0.021021%
BE  | Solar change: -0